# TriFactAI · Stage 1–3 в Colab

Запустите в **hosted runtime** Colab. Для GPU-задания выберите NVIDIA L4, для DVC стадий и отчётов — CPU. Убедитесь, что доступный профиль укладывается в утверждённый бюджет. Первая ячейка подключает приватный Google Drive. Контракт и порядок проверки описаны в ADR-0025.


In [ ]:
import json
import os
import re
import shutil
import subprocess
from pathlib import Path

from google.colab import drive

DRIVE = Path("/content/drive/MyDrive/TrifactAI")
WORK = Path("/content/TrifactAI")
SOURCE = "https://github.com/Nusultan11/TriFactAI.git"  # либо путь к проверенному .bundle в Drive
REVISION = ""  # полный SHA нужного коммита

drive.mount("/content/drive")
assert (DRIVE / "dvc/files/md5").is_dir(), "DVC cache is missing from Drive"
assert re.fullmatch(r"[0-9a-f]{40}", REVISION), "Set the 40-character Git SHA"


def run(*args, **kwargs):
    return subprocess.run([str(arg) for arg in args], check=True, **kwargs)


if not WORK.exists():
    run("git", "clone", "--no-checkout", SOURCE, WORK)
run("git", "-C", WORK, "checkout", "--detach", REVISION)
os.chdir(WORK)
run("python", "-m", "pip", "install", "uv==0.12.1")
run("uv", "sync", "--locked", "--extra", "ml", "--extra", "dev")
UV = (shutil.which("uv"), "run", "--no-sync")
DVC = (*UV, "dvc")
run(*DVC, "remote", "add", "--local", "--force", "--default", "drive", DRIVE / "dvc")
run(*DVC, "remote", "modify", "--local", "drive", "verify", "true")
run(*DVC, "config", "--local", "core.autostage", "false")

## Восстановление и инвентаризация

Запустите на чистой сессии. Для крупных стадий указывайте целевые выходы и проверяйте свободный диск до выгрузки. Размер всего кеша больше пространства отдельных Colab runtime.


In [ ]:
def verify_dvc(*targets, cloud=False):
    remote_args = ("--cloud", "--remote", "drive") if cloud else ()
    result = run(*DVC, "status", "--json", *remote_args, *targets, capture_output=True, text=True)
    changes = json.loads(result.stdout)
    if changes:
        raise RuntimeError(f"DVC differences for {targets}: {changes}")


RESTORE_TARGETS = ("data/review.dvc", "data/external.dvc")
run(*DVC, "pull", *RESTORE_TARGETS)
verify_dvc(*RESTORE_TARGETS, cloud=True)
verify_dvc(*RESTORE_TARGETS)

run(
    *UV,
    "python",
    "scripts/restore_model.py",
    DRIVE / "models/lid176/lid.176.bin.parts.json",
    "--dest",
    WORK / "data/.cache/models/lid176",
)

## MLflow для текущего облачного сеанса

SQLite работает на диске VM. Артефакты MLflow записываются в Drive. Для единого Registry используйте существующий авторизованный облачный MLflow. Этот профиль сохраняет отдельную базу на каждую сессию и восстанавливается по выбранному снимку. Снимки создаются после каждого задания.


In [ ]:
import sqlite3
import time
import urllib.request
from uuid import uuid4

TRACKING = DRIVE / "workspace-data/mlflow"
TRACKING.mkdir(parents=True, exist_ok=True)
SESSION = uuid4().hex
DB = Path("/content") / f"trifact-{SESSION}.db"
RESTORE_DB = None  # путь к конкретному предыдущему *.db в TRACKING для восстановления
if RESTORE_DB:
    shutil.copyfile(RESTORE_DB, DB)

with open(f"/content/mlflow-{SESSION}.log", "w") as log:
    server = subprocess.Popen(
        [
            *UV,
            "mlflow",
            "server",
            "--host",
            "127.0.0.1",
            "--port",
            "5000",
            "--backend-store-uri",
            f"sqlite:///{DB}",
            "--no-serve-artifacts",
            "--default-artifact-root",
            (TRACKING / "artifacts").as_uri(),
        ],
        stdout=log,
        stderr=subprocess.STDOUT,
    )
os.environ["MLFLOW_TRACKING_URI"] = "http://127.0.0.1:5000"
for _ in range(60):
    if server.poll() is not None:
        raise RuntimeError(f"MLflow failed; see /content/mlflow-{SESSION}.log")
    try:
        with urllib.request.urlopen(os.environ["MLFLOW_TRACKING_URI"] + "/health", timeout=1):
            break
    except OSError:
        time.sleep(1)
else:
    raise TimeoutError("MLflow did not become healthy")


def snapshot_mlflow():
    import hashlib
    from contextlib import closing

    temp_db = DB.with_suffix(".snapshot")
    snapshot = TRACKING / f"{SESSION}-{time.time_ns()}.db"
    pending = snapshot.with_suffix(".tmp")

    def digest(path):
        with path.open("rb") as stream:
            return hashlib.file_digest(stream, "sha256").hexdigest()

    with closing(sqlite3.connect(DB)) as source, closing(sqlite3.connect(temp_db)) as dest:
        source.backup(dest)
        assert dest.execute("PRAGMA integrity_check").fetchone() == ("ok",)
    shutil.copyfile(temp_db, pending)
    assert digest(temp_db) == digest(pending), "MLflow snapshot changed during copy"
    pending.replace(snapshot)
    print(snapshot)


def run_logged(*args):
    try:
        run(*args)
    finally:
        snapshot_mlflow()

## CPU-стадия DVC

Подтяните необходимые входы для выбранной стадии. В Colab используйте отдельную копию репозитория и целевой профиль; полный граф имеет собственные требования к диску и времени. Перед публикацией результатов проверьте манифесты и изменения `dvc.lock`.


In [ ]:
STAGE = ""  # например eda_report после восстановления её входов
STAGE_INPUTS = []  # нужные DVC-цели входов выбранной стадии
if STAGE:
    if STAGE_INPUTS:
        run(*DVC, "pull", *STAGE_INPUTS)
    run_logged(*DVC, "repro", "--single-item", STAGE)
    run(*DVC, "push", STAGE)
    verify_dvc(STAGE, cloud=True)

## GPU-шард

Подготовьте каталог с одним ограниченным по размеру набором Parquet и `config.json`. Сначала выполните пилот. Для перевода задайте `revision` и `back_revision` из `params.yaml: mt`. Учитель использует L4-профиль, переопределения можно передать через объект `colab` в конфиге. Результат находится в `out/` конкретной попытки. Выходы разных попыток и шардов импортируйте отдельно или объединяйте с уникальными именами файлов и проверкой идентификаторов.


In [ ]:
JOB = ""  # teacher, translate или nli_score
INPUTS = None  # Path к подготовленному шарду под DRIVE или /content
TIMEOUT_MINUTES = 45
if JOB:
    assert INPUTS is not None
    run_logged(
        *UV,
        "python",
        "-m",
        "trifact.batch.colab",
        JOB,
        "--inputs",
        INPUTS,
        "--drive",
        DRIVE,
        "--timeout-minutes",
        TIMEOUT_MINUTES,
    )

## Git backup

Снимок содержит историю Git и указатели DVC. Перед запуском опубликуйте проверенные указатели данных и проверьте наличие соответствующих объектов в Drive.


In [ ]:
BACKUP_TAG = None
if BACKUP_TAG:
    run(
        *UV,
        "python",
        "scripts/backup.py",
        "--tag",
        BACKUP_TAG,
        "--dest",
        DRIVE / "backups",
        "--git-only",
        "--keep",
        "0",
    )